# GP1 ASR - CRDNN Local Dry-run (macOS / MPS)

Local end-to-end pipeline

## 1. User-editable paths

In [1]:
from pathlib import Path

DATA_ROOT = Path("/Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/data")
GP1_ROOT = Path("/Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1")

print("GP1 root :", GP1_ROOT)
print("DATA_ROOT:", DATA_ROOT)
for p in sorted(DATA_ROOT.iterdir())[:20]:
    print(p.name)

GP1 root : /Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1
DATA_ROOT: /Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/data
dev
test
train


## 2. Device detection

In [2]:
import torch

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print("torch :", torch.__version__)
print("device:", DEVICE)

torch : 2.5.1
device: mps


In [5]:
import sys, os
print("executable:", sys.executable)
print("prefix:    ", sys.prefix)
print("version:   ", sys.version)
print("cwd:       ", os.getcwd())
print("---sys.path---")
for p in sys.path:
    print(p)

executable: /Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/.venv/bin/python
prefix:     /Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/.venv
version:    3.11.15 (main, Apr 14 2026, 14:45:51) [Clang 22.1.3 ]
cwd:        /Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/group-projects/gp1/notebooks
---sys.path---
/Users/deniskazhekin/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python311.zip
/Users/deniskazhekin/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11
/Users/deniskazhekin/.local/share/uv/python/cpython-3.11-macos-aarch64-none/lib/python3.11/lib-dynload

/Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/.venv/lib/python3.11/site-packages


In [ ]:
!

/Users/deniskazhekin/Downloads/ITMO_Speech_Recognition_Course/.venv/bin/python


## 3. Resolve train / dev / test paths

We use the course-provided splits directly:
- `data/train/` — 6 speakers, ~12.5k samples (training)
- `data/dev/`   — 10 speakers, ~2.3k samples (early stopping + HP search)
- `data/test/`  — 14 speakers, ~2.6k samples (Kaggle submission, no labels)

No random 90/10 carving — the course's dev/ is already a proper OOD holdout.

In [ ]:
# Use the real course-provided train/ and dev/ splits.
# - train/ = 6 speakers, 12,553 samples
# - dev/   = 10 speakers, 2,265 samples (OOD for 4 of them)

TRAIN_CSV      = DATA_ROOT / "train" / "train.csv"
TRAIN_WAV_ROOT = DATA_ROOT / "train"
DEV_CSV        = DATA_ROOT / "dev" / "dev.csv"
DEV_WAV_ROOT   = DATA_ROOT / "dev"
TEST_CSV       = DATA_ROOT / "test" / "test.csv"
TEST_WAV_ROOT  = DATA_ROOT / "test"

for p in [TRAIN_CSV, TRAIN_WAV_ROOT, DEV_CSV, DEV_WAV_ROOT, TEST_CSV, TEST_WAV_ROOT]:
    assert p.exists(), f"Missing: {p}"

import csv as _csv
with open(TRAIN_CSV, encoding="utf-8") as fh:
    n_train = sum(1 for _ in _csv.DictReader(fh))
with open(DEV_CSV, encoding="utf-8") as fh:
    n_dev = sum(1 for _ in _csv.DictReader(fh))
print(f"train: {n_train} samples -> {TRAIN_CSV}")
print(f"dev:   {n_dev} samples -> {DEV_CSV}")


## HP Search (Random, N=10)

Random search over a 5-dimensional hyperparameter space following Bergstra & Bengio (2012) JMLR — 10 independent trials (smaller budget for local, where each trial runs on CPU/MPS without GPU acceleration). The formula N >= log(1-p)/log(1-alpha) guarantees p=0.95 probability of finding a configuration in the top alpha fraction of the search space with N trials.

In [ ]:
import math, json, yaml, sys

BASELINE = "crdnn"
CONFIG_NAME = "crdnn.yaml"
N_TRIALS = 10
RANDOM_SEED = 42

OUTPUT_BASE = GP1_ROOT / "_local_runs" / "crdnn"
NUM_WORKERS = 2

# Make gp1 package importable for config resolution.
_gp1_src = str(GP1_ROOT / "src")
if _gp1_src not in sys.path:
    sys.path.insert(0, _gp1_src)

# Shared config loader — mirrors scripts/train.py semantics exactly.
# Resolves `defaults:` inheritance via gp1/config.py (single source of truth).
from gp1.config import load_config as _load_config  # noqa: E402


def sample_params(rng: random.Random) -> dict:
    """5-dim search space per Bergstra & Bengio (2012)."""
    return {
        "lr": 10 ** rng.uniform(-4, math.log10(5e-3)),
        "dropout": rng.uniform(0.05, 0.25),
        "warmup_steps": rng.randint(500, 8000),
        "grad_clip_norm": rng.choice([0.5, 1.0, 2.0, 5.0]),
        "specaug_time_mask_param": rng.choice([15, 25, 35, 50]),
    }


def patch_config(base_path: Path, params: dict, out_path: Path) -> None:
    """Load base config, merge defaults, apply HP params, write patched YAML.

    Defaults resolution delegates to ``gp1.config.load_config`` (the shared
    helper extracted from scripts/train.py). HP-parameter overriding is
    done inline here to keep the notebook cell self-contained.
    """
    # Resolve `defaults:` inheritance via the shared single-source-of-truth.
    cfg = _load_config(base_path)
    cfg.setdefault("train", {})
    cfg["train"]["lr"] = params["lr"]
    cfg["train"].setdefault("optimizer", {})
    cfg["train"]["optimizer"]["lr"] = params["lr"]
    cfg["train"].setdefault("scheduler", {})
    cfg["train"]["scheduler"]["warmup_steps"] = params["warmup_steps"]
    cfg["train"]["grad_clip_norm"] = params["grad_clip_norm"]
    cfg.setdefault("model", {})
    cfg["model"]["dropout"] = params["dropout"]
    cfg.setdefault("aug", {})
    cfg["aug"]["specaug_time_mask_param"] = params["specaug_time_mask_param"]
    with open(out_path, "w") as f:
        yaml.safe_dump(cfg, f)

In [ ]:
import os
rng = random.Random(RANDOM_SEED)
trials_dir = OUTPUT_BASE / "hp_search"
trials_dir.mkdir(parents=True, exist_ok=True)
trials_log = trials_dir / "trials.jsonl"
best = {"trial_id": None, "cer": float("inf"), "ckpt": None, "params": None}

# Ensure tqdm refreshes visibly in Jupyter (non-tty).
os.environ.setdefault("TQDM_MININTERVAL", "0.5")
os.environ.setdefault("PYTHONUNBUFFERED", "1")

for trial_id in range(N_TRIALS):
    params = sample_params(rng)
    trial_dir = trials_dir / f"trial_{trial_id:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    patched = trial_dir / "config.yaml"
    patch_config(GP1_ROOT / "configs" / CONFIG_NAME, params, patched)
    cmd = [
        sys.executable, "-u", "scripts/train.py",
        "--config", str(patched),
        "--train-csv", str(TRAIN_CSV), "--train-root", str(TRAIN_WAV_ROOT),
        "--dev-csv", str(DEV_CSV), "--dev-root", str(DEV_WAV_ROOT),
        "--output-dir", str(trial_dir),
        "--num-workers", str(NUM_WORKERS),
        "--device", DEVICE,
    ]
    print(f"\n===== trial {trial_id:03d}  params={params} =====")

    stderr_log_path = trial_dir / "stderr.log"
    import subprocess
    proc = subprocess.Popen(
        cmd,
        cwd=str(GP1_ROOT),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )

    # Forward stdout line-by-line (progress bars, INFO logs).
    stderr_tail: list[str] = []
    with open(stderr_log_path, "w") as stderr_fh:
        import threading

        def _pipe_stderr():
            for line in proc.stderr:
                stderr_fh.write(line)
                stderr_fh.flush()
                stderr_tail.append(line)
                if len(stderr_tail) > 200:
                    stderr_tail.pop(0)
                # Mirror WARN/ERROR lines live to keep user informed.
                if " ERROR " in line or " WARNING " in line or "Traceback" in line:
                    print(line, end="")

        t = threading.Thread(target=_pipe_stderr, daemon=True)
        t.start()
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        t.join(timeout=2.0)

    if proc.returncode != 0:
        print(f"\ntrial {trial_id:03d}: FAILED (rc={proc.returncode})")
        print("--- stderr (last 25 lines) ---")
        print("".join(stderr_tail[-25:]))
        cer = float("inf")
        result = {"error": f"rc={proc.returncode}"}
    else:
        try:
            result = json.loads((trial_dir / "result.json").read_text())
            cer = float(result["best_val_cer"])
        except Exception as exc:
            print(f"\ntrial {trial_id:03d}: could not parse result.json: {exc}")
            cer = float("inf")
            result = {"error": str(exc)}

    with open(trials_log, "a") as f:
        f.write(json.dumps({"trial_id": trial_id, "params": params, "best_val_cer": cer}) + "\n")
    if cer < best["cer"]:
        best = {
            "trial_id": trial_id,
            "cer": cer,
            "ckpt": result.get("best_ckpt_path") if isinstance(result, dict) else None,
            "params": params,
        }
    print(f"trial {trial_id:03d}: CER={cer:.4f}  lr={params['lr']:.2e}")

print(f"\nBEST trial_id={best['trial_id']} CER={best['cer']:.4f}  ckpt={best['ckpt']}")
(trials_dir / "best_trial.json").write_text(json.dumps(best, indent=2, default=str))


## 4. Predict on test data

In [ ]:
# Slice 32 rows from test.csv for a quick inference pass.
TEST_TINY_CSV = SPLIT_DIR / "test_tiny.csv"
with open(TEST_CSV, encoding="utf-8") as fh:
    test_rows = list(csv.DictReader(fh))[:32]
with open(TEST_TINY_CSV, "w", encoding="utf-8", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=list(test_rows[0].keys()))
    w.writeheader()
    w.writerows(test_rows)

SUBMISSION_CSV = trials_dir / "submission.csv"
cmd = [
    sys.executable, "scripts/predict.py",
    "--checkpoint", best["ckpt"],
    "--config", str(trials_dir / f"trial_{best['trial_id']:03d}" / "config.yaml"),
    "--test-csv", str(TEST_TINY_CSV), "--test-root", str(TEST_WAV_ROOT),
    "--output", str(SUBMISSION_CSV),
    "--device", DEVICE,
]
print("$", " ".join(cmd))
subprocess.run(cmd, cwd=str(GP1_ROOT), check=True)
print("Wrote:", SUBMISSION_CSV)

## 5. Preview

In [ ]:
import pandas as pd
df = pd.read_csv(SUBMISSION_CSV)
print(df.shape)
df.head(10)